# Day 10: Mall Customer Dataset Analysis (Ayan's Version)
EDA and customer behavior analysis using the Mall Customers dataset (`Mall_Customers.csv`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="rocket")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12


### Step 1: Load Dataset


In [ ]:
df = pd.read_csv("../../DataSet/Mall_Customers.csv")
print("Mall Customer dataset loaded!")


### Step 2: Check Shape


In [ ]:
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.shape


### Step 3: View Dataset Information


In [ ]:
df.head()


In [ ]:
df.tail()


### Step 4: Check Data Types


In [ ]:
df.info()


### Step 5: Identify Missing Values


In [ ]:
df.isnull().sum()


### Step 6: Handle Missing Values


In [ ]:
print("No missing values to handle.")


### Step 7: Check Duplicate Records


In [ ]:
duplicates = df.duplicated().sum()
print("Duplicates count:", duplicates)
if duplicates > 0:
    display(df[df.duplicated(keep=False)])


### Step 8: Remove Duplicates


In [ ]:
# Removing duplicates (if any)
df.drop_duplicates(inplace=True)


### Step 9: Generate Statistical Summary


In [ ]:
df.describe()


### Step 10: Outlier Detection


In [ ]:
# Check Boxplot for Annual Income to detect outliers
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["Annual Income (k$)"], color="salmon")
plt.title("Boxplot of Annual Income (k$)")
plt.xlabel("Annual Income (k$)")
plt.show()

# Calculating IQR to see the exact outlier boundaries
Q1 = df["Annual Income (k$)"].quantile(0.25)
Q3 = df["Annual Income (k$)"].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR
print(f"Outlier limits: Lower = {lower_limit}, Upper = {upper_limit}")
print("Outliers count:", len(df[df["Annual Income (k$)"] > upper_limit]))


### Step 11: Perform Univariate Analysis


In [ ]:
# Fix the user's syntax error: plt.bar(df["Annual Income (k$)"].value_counts(), height=True)
# We replace it with a clean distribution histogram of Annual Income

plt.figure(figsize=(8, 5))
sns.histplot(df["Annual Income (k$)"], kde=True, color="teal")
plt.title("Distribution of Annual Income (k$)")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Count")
plt.show()


### Step 12: Perform Bivariate Analysis


In [ ]:
# Plot Annual Income vs Spending Score
plt.figure(figsize=(9, 6))
sns.scatterplot(x="Annual Income (k$)", y="Spending Score (1-100)", data=df, hue="Genre", s=80, alpha=0.9)
plt.title("Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Gender")
plt.show()


In [ ]:
# Target Group Identification (Middle Income, High Spending Customers)
target_group = df[(df['Annual Income (k$)'] >= 40) & 
                  (df['Annual Income (k$)'] <= 70) & 
                  (df['Spending Score (1-100)'] > 60)]

print(f"Target group shape: {target_group.shape}")
target_group.head()


In [ ]:
# Visualize Target Group
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', 
                data=target_group, color='red', label='Target Group (Middle Inc, High Spend)', s=100)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Middle-Income High-Spending Customers")
plt.legend()
plt.show()


### Step 13: Correlation Analysis


In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.show()


### Step 14: Feature Engineering


In [ ]:
# Let's bin Ages to understand spending habits of middle-income customers
middle_income = df[(df['Annual Income (k$)'] >= 40) & (df['Annual Income (k$)'] <= 70)].copy()

middle_income['AgeGroup'] = pd.cut(middle_income['Age'], 
                                   bins=[0,20,30,40,50,60,70], 
                                   labels=['<20','20-30','30-40','40-50','50-60','60+'])

age_spending = middle_income.groupby('AgeGroup', observed=False)['Spending Score (1-100)'].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(x="AgeGroup", y="Spending Score (1-100)", data=age_spending, palette="Oranges")
plt.xlabel("Age Group (Middle Income)")
plt.ylabel("Average Spending Score")
plt.title("Average Spending Score by Age Group (Middle Income Customers)")
plt.show()


In [ ]:
# Let's bin Income Groups to analyze overall spending differences
df_eng = df.copy()
df_eng['IncomeGroup'] = pd.cut(df_eng['Annual Income (k$)'], 
                            bins=[0,40,70,150], 
                            labels=['Low Income','Middle Income','High Income'])

income_spending = df_eng.groupby('IncomeGroup', observed=False)['Spending Score (1-100)'].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(x="IncomeGroup", y="Spending Score (1-100)", data=income_spending, palette="viridis")
plt.xlabel("Income Group")
plt.ylabel("Average Spending Score")
plt.title("Average Spending Score by Income Group")
plt.show()


### Step 15: Draw Conclusions and Insights
1. **Five Customer Segments**: The scatter plot of Annual Income vs Spending Score shows 5 distinct clusters:
   * Low Income, Low Spend
   * Low Income, High Spend
   * Mid Income, Mid Spend (average spending score ~50)
   * High Income, Low Spend
   * High Income, High Spend
2. **Age Group Dynamics**: Among middle-income customers, the 20-30 and 30-40 age groups show the highest average spending scores, suggesting marketing campaigns should target younger individuals.
3. **Outliers**: There is one minor outlier on the upper end of annual income (above 137k$), which falls outside the standard 1.5 * IQR boundary.
